# ☀️ Energybae — Solar Load Calculator
### Electricity Bill → AI Extraction → Solar Recommendation → Excel Report

**AI Intern Practical Task | Energybae, Pimpri, Pune**

---

### How to run this notebook

| Step | What to do |
|------|------------|
| **Step 1** | Install packages *(once per Colab session)* |
| **Step 2** | Get a **free** Gemini API key and add it to Colab Secrets |
| **Step 3** | Run once to define all functions |
| **Step 4** | Upload your electricity bill (PDF / JPG / PNG) |
| **Step 5** | View results and download the filled Excel file |

> **No payment required.** This notebook uses the Google Gemini API which has a free tier — no credit card needed.

**Supported bills:** MSEDCL, BESCOM, TATA Power, CESC, Adani Electricity, and other Indian utilities.

---

## Step 1 — Install required packages
Run this cell once per Colab session. It takes about 30 seconds.

In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'google-genai', 'pdfplumber', 'Pillow', 'openpyxl'
])
print('✅ All packages installed successfully.')

## Step 2 — Get a free Gemini API key

**How to get your free key (2 minutes):**
1. Go to **https://aistudio.google.com/app/apikey**
2. Sign in with any Google account
3. Click **"Create API Key"** — it is free, no credit card needed
4. Copy the key (starts with `AIza...`)

**How to add it to Colab:**
1. Click the 🔑 **Secrets** icon in the left sidebar of Colab
2. Click **"+ Add new secret"**
3. Name: `GEMINI_API_KEY` — Value: paste your key
4. Toggle **"Notebook access"** ON
5. Run the cell below to verify

In [ ]:
import os

# ── Option A (recommended): Colab Secrets — key stays private ──────────────
try:
    from google.colab import userdata
    _key = userdata.get('GEMINI_API_KEY')
    if _key:
        os.environ['GEMINI_API_KEY'] = _key
        print('✅ Gemini API key loaded from Colab Secrets.')
    else:
        raise KeyError('not set')
except Exception:
    # ── Option B: paste key directly (remove before sharing this notebook!) ─
    # os.environ['GEMINI_API_KEY'] = 'AIza...'   # ← uncomment and paste here
    if os.environ.get('GEMINI_API_KEY'):
        print('✅ Gemini API key found in environment.')
    else:
        print('⚠️  No API key found!')
        print('   Follow the instructions in Step 2 above to add your free key.')
        print()
        print('   Get your free key at: https://aistudio.google.com/app/apikey')

## Step 3 — Define all helper functions
Run this cell once. It sets up the AI extractor, solar calculator, and Excel generator — all in one place.

In [ ]:
"""
═══════════════════════════════════════════════════════════════
  ENERGYBAE — SOLAR LOAD CALCULATOR  (self-contained)
  Uses Google Gemini (free tier) for AI bill extraction.
  No payment required.
═══════════════════════════════════════════════════════════════
"""

import os, io, json, re, math
from google import genai
from google.genai import types
from PIL import Image
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter


# ════════════════════════════════════════════════════════════════════════════
#  PART 1 — AI-POWERED BILL EXTRACTOR  (Google Gemini)
# ════════════════════════════════════════════════════════════════════════════

EXTRACTION_PROMPT = """You are an expert at reading Indian electricity bills from
utilities like MSEDCL, BESCOM, TATA Power, CESC, Adani Electricity, etc.

Extract ONLY these fields from the bill:
- Consumer Name
- Consumer Number / Account Number
- Billing Month (format: "Month YYYY", e.g., "March 2025")
- Units Consumed in kWh — total units this billing period
  (look for: Units consumed, Total units, Net units)
- Sanctioned Load in kW — contracted/maximum allowed load
  (look for: Sanctioned load, Connected load, Contract demand)
- Tariff Category — slab or code (e.g., LT-I, LT-II, Domestic, Commercial)
- Total Bill Amount in INR — the final payable amount
- Meter Number — meter serial number if visible
- Distribution Company — name of electricity provider

Rules:
- Return ONLY a valid JSON object. No explanation, no markdown.
- Use null for any field you cannot find.
- All numbers must be numeric (not strings).
- If units consumed appears multiple times, use the NET or TOTAL value.

Return exactly this JSON structure:
{"consumer_name":"","consumer_number":"","billing_month":"","units_consumed":0,
 "sanctioned_load":0,"tariff_category":"","total_bill_amount":0,
 "meter_number":null,"distribution_company":null}"""


GEMINI_MODEL = 'gemini-2.0-flash'   # free tier

def _get_client():
    api_key = os.environ.get('GEMINI_API_KEY')
    if not api_key:
        raise ValueError(
            'Gemini API key not found.\n'
            'Run Step 2 to add your free key from https://aistudio.google.com/app/apikey'
        )
    return genai.Client(api_key=api_key)


def _parse_json(content: str) -> dict:
    """Robustly extract JSON from Gemini's response."""
    try:
        return json.loads(content.strip())
    except json.JSONDecodeError:
        pass
    # Strip markdown code fences
    match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', content, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except json.JSONDecodeError:
            pass
    # Find the first {...} block
    match = re.search(r'\{[\s\S]*\}', content)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            pass
    raise ValueError(f'Could not parse JSON from AI response:\n{content[:500]}')


def _sanitise(raw: dict) -> dict:
    def to_float(val, default=0.0):
        try:
            return float(str(val).replace(',', '').strip())
        except (TypeError, ValueError):
            return default
    return {
        'consumer_name':        str(raw.get('consumer_name') or 'Unknown'),
        'consumer_number':      str(raw.get('consumer_number') or 'Unknown'),
        'billing_month':        str(raw.get('billing_month') or 'Unknown'),
        'units_consumed':       to_float(raw.get('units_consumed')),
        'sanctioned_load':      to_float(raw.get('sanctioned_load')),
        'tariff_category':      str(raw.get('tariff_category') or 'Unknown'),
        'total_bill_amount':    to_float(raw.get('total_bill_amount')),
        'meter_number':         str(raw['meter_number']) if raw.get('meter_number') else None,
        'distribution_company': str(raw['distribution_company']) if raw.get('distribution_company') else None,
    }


def extract_from_pdf(pdf_bytes: bytes) -> dict:
    """Extract bill data from a text-based PDF."""
    import pdfplumber
    raw_text = ''
    with pdfplumber.open(io.BytesIO(pdf_bytes)) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                raw_text += text + '\n'
    if not raw_text.strip() or len(raw_text.strip()) < 30:
        raise ValueError(
            'This PDF appears to be scanned (image-only, no text layer). '
            'Please export or screenshot the bill as a JPG/PNG and upload that.'
        )
    client = _get_client()
    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=f'{EXTRACTION_PROMPT}\n\nElectricity bill text:\n{raw_text}'
    )
    return _sanitise(_parse_json(response.text))


def extract_from_image(image_bytes: bytes, filename: str = '') -> dict:
    """Extract bill data from an image using Gemini Vision."""
    mime = 'image/png' if filename.lower().endswith('.png') else 'image/jpeg'
    client = _get_client()
    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=[
            types.Part.from_bytes(data=image_bytes, mime_type=mime),
            EXTRACTION_PROMPT,
        ]
    )
    return _sanitise(_parse_json(response.text))


def extract_bill_data(file_bytes: bytes, filename: str) -> dict:
    """
    Main entry point. Detects file type and extracts bill data using Gemini AI.
    Supports: PDF (text-based), JPG, JPEG, PNG
    """
    lower = filename.lower()
    if lower.endswith('.pdf'):
        return extract_from_pdf(file_bytes)
    elif lower.endswith(('.jpg', '.jpeg', '.png')):
        return extract_from_image(file_bytes, filename)
    else:
        raise ValueError(f'Unsupported file: {filename}. Please use PDF, JPG, or PNG.')


# ════════════════════════════════════════════════════════════════════════════
#  PART 2 — SOLAR CALCULATOR
# ════════════════════════════════════════════════════════════════════════════

PEAK_SUN_HOURS       = 4.5      # India average peak sun hours/day
SOLAR_GEN_FACTOR     = 4.0      # kWh generated per installed kW per day
SYSTEM_COST_PER_KWP  = 60_000   # INR per kWp (approximate installed cost)
CO2_FACTOR           = 0.82     # kg CO₂ per kWh (India grid, CEA 2023)
SYSTEM_LIFE_YEARS    = 25


def calculate_solar_recommendation(bill_data: dict) -> dict:
    units  = float(bill_data.get('units_consumed')  or 0)
    amount = float(bill_data.get('total_bill_amount') or 0)
    s_load = float(bill_data.get('sanctioned_load')  or 0)

    daily_units = units / 30
    # Size the system to cover consumption OR 80% of sanctioned load, whichever is larger
    raw_size       = max(daily_units / PEAK_SUN_HOURS, s_load * 0.8)
    recommended_kw = math.ceil(raw_size * 2) / 2 if raw_size > 0 else 1.0

    cost_per_unit  = (amount / units) if units > 0 else 7.0
    monthly_gen    = recommended_kw * SOLAR_GEN_FACTOR * 30
    covered_units  = min(monthly_gen, units)
    monthly_saving = round(covered_units * cost_per_unit)
    annual_saving  = monthly_saving * 12
    system_cost    = recommended_kw * SYSTEM_COST_PER_KWP
    payback_years  = round(system_cost / annual_saving, 1) if annual_saving > 0 else 0.0
    annual_gen     = monthly_gen * 12

    return {
        'recommended_system_size_kw': recommended_kw,
        'estimated_monthly_savings':  monthly_saving,
        'estimated_annual_savings':   annual_saving,
        'payback_period_years':       payback_years,
        'co2_reduction_kg_per_year':  round(annual_gen * CO2_FACTOR),
        'system_cost_inr':            round(system_cost),
        'lifetime_savings_inr':       annual_saving * SYSTEM_LIFE_YEARS,
        'net_benefit_inr':            (annual_saving * SYSTEM_LIFE_YEARS) - system_cost,
        'daily_units':                round(daily_units, 2),
        'monthly_generation_kwh':     round(monthly_gen, 1),
        'cost_per_unit':              round(cost_per_unit, 2),
    }


# ════════════════════════════════════════════════════════════════════════════
#  PART 3 — EXCEL REPORT GENERATOR
# ════════════════════════════════════════════════════════════════════════════

def _fill(hex_color):    return PatternFill('solid', fgColor=hex_color)
def _thin_border():
    t = Side(style='thin', color='CCCCCC')
    return Border(top=t, bottom=t, left=t, right=t)

def _section_header(ws, row, title):
    ws.merge_cells(f'A{row}:F{row}')
    c = ws[f'A{row}']
    c.value = title
    c.font = Font(bold=True, size=12, color='FFFFFF')
    c.fill = _fill('1A5276')
    c.alignment = Alignment(horizontal='left', vertical='middle', indent=1)
    ws.row_dimensions[row].height = 24

def _data_row(ws, row, label, value, unit=''):
    lc = ws.cell(row=row, column=1, value=label)
    lc.fill = _fill('F0F0F0')
    lc.border = _thin_border()
    ws.merge_cells(f'B{row}:D{row}')
    vc = ws.cell(row=row, column=2, value=value)
    vc.border = _thin_border()
    if unit:
        ws.merge_cells(f'E{row}:F{row}')
        uc = ws.cell(row=row, column=5, value=unit)
        uc.font = Font(color='777777', italic=True)
    ws.row_dimensions[row].height = 22

def _fmt_inr(n):
    try:
        n = int(round(n))
        s = str(abs(n))
        result = s[-3:]
        s = s[:-3]
        while s:
            result = s[-2:] + ',' + result
            s = s[:-2]
        return ('₹ -' if n < 0 else '₹ ') + result.lstrip(',')
    except Exception:
        return f'₹ {n}'


def generate_excel(bill_data: dict, solar: dict) -> bytes:
    """Generate a formatted Excel workbook and return as bytes."""
    wb = Workbook()
    ws = wb.active
    ws.title = 'Solar Load Calculator'

    for i, w in enumerate([30, 22, 16, 16, 18, 16], 1):
        ws.column_dimensions[get_column_letter(i)].width = w

    # Title
    ws.merge_cells('A1:F1')
    t = ws['A1']
    t.value = 'ENERGYBAE — Solar Load Calculator'
    t.font = Font(bold=True, size=16, color='FFFFFF')
    t.fill = _fill('2C7A2C')
    t.alignment = Alignment(horizontal='center', vertical='middle')
    ws.row_dimensions[1].height = 36

    ws.merge_cells('A2:F2')
    s = ws['A2']
    s.value = 'Electricity Bill Analysis & Solar System Recommendation'
    s.font = Font(italic=True, size=11, color='555555')
    s.alignment = Alignment(horizontal='center')
    ws.row_dimensions[2].height = 20

    _section_header(ws, 4, 'SECTION 1 — Customer Information')
    _data_row(ws, 5,  'Consumer Name',        bill_data.get('consumer_name', 'N/A'))
    _data_row(ws, 6,  'Consumer Number',       bill_data.get('consumer_number', 'N/A'))
    _data_row(ws, 7,  'Meter Number',          bill_data.get('meter_number') or 'N/A')
    _data_row(ws, 8,  'Distribution Company',  bill_data.get('distribution_company') or 'N/A')
    _data_row(ws, 9,  'Billing Month',         bill_data.get('billing_month', 'N/A'))
    _data_row(ws, 10, 'Tariff Category',       bill_data.get('tariff_category', 'N/A'))

    _section_header(ws, 12, 'SECTION 2 — Electricity Usage')
    _data_row(ws, 13, 'Units Consumed',             bill_data.get('units_consumed', 0),              'kWh / month')
    _data_row(ws, 14, 'Sanctioned Load',             bill_data.get('sanctioned_load', 0),            'kW')
    _data_row(ws, 15, 'Total Bill Amount',           _fmt_inr(bill_data.get('total_bill_amount', 0)))
    _data_row(ws, 16, 'Cost per Unit',               f"₹ {solar.get('cost_per_unit', 0):.2f}",       '₹ / kWh')
    _data_row(ws, 17, 'Average Daily Consumption',   f"{solar.get('daily_units', 0):.2f}",           'kWh / day')

    _section_header(ws, 19, 'SECTION 3 — Solar System Recommendation')
    rows = [
        (20, 'Recommended System Size',   solar.get('recommended_system_size_kw', 0), 'kWp'),
        (21, 'Estimated Monthly Savings', _fmt_inr(solar.get('estimated_monthly_savings', 0)), ''),
        (22, 'Estimated Annual Savings',  _fmt_inr(solar.get('estimated_annual_savings', 0)), ''),
        (23, 'Estimated Payback Period',  solar.get('payback_period_years', 0), 'years'),
        (24, 'CO₂ Reduction',             f"{solar.get('co2_reduction_kg_per_year', 0):,}", 'kg CO₂ / year'),
    ]
    for row, label, value, unit in rows:
        _data_row(ws, row, label, value, unit)
        if row in (20, 21, 22):
            ws.cell(row=row, column=1).font = Font(bold=True)
            ws.cell(row=row, column=2).font = Font(bold=True)

    _section_header(ws, 26, 'SECTION 4 — Financial Summary (25-year projection)')
    _data_row(ws, 27, 'Estimated System Cost',     _fmt_inr(solar.get('system_cost_inr', 0)),     '(approx. ₹60,000/kWp installed)')
    _data_row(ws, 28, '25-Year Savings',            _fmt_inr(solar.get('lifetime_savings_inr', 0)))
    _data_row(ws, 29, 'Net Benefit (25yr − Cost)',  _fmt_inr(solar.get('net_benefit_inr', 0)))

    ws.merge_cells('A31:F31')
    f1 = ws['A31']
    f1.value = 'Generated by Energybae Solar Load Calculator | www.energybae.in | energybae.co@gmail.com'
    f1.font = Font(italic=True, size=9, color='888888')
    f1.alignment = Alignment(horizontal='center')

    ws.merge_cells('A32:F32')
    f2 = ws['A32']
    f2.value = '* Estimates based on 4.5 peak sun hours/day (India avg.) and ₹60,000/kWp installation cost. Actual results may vary.'
    f2.font = Font(italic=True, size=8, color='AAAAAA')
    f2.alignment = Alignment(horizontal='center')

    buf = io.BytesIO()
    wb.save(buf)
    return buf.getvalue()


print('✅ All functions ready.')
print('   extract_bill_data()              → AI extraction via Gemini (free)')
print('   calculate_solar_recommendation() → Solar sizing & savings')
print('   generate_excel()                 → Formatted Excel report')

## Step 4 — Upload your electricity bill

A file picker will appear. Select your bill (PDF, JPG, or PNG).  
AI extraction takes about **5–15 seconds**.

In [ ]:
from google.colab import files as colab_files

print('📁 A file picker will appear below — select your electricity bill.')
uploaded = colab_files.upload()

if not uploaded:
    print('❌ No file selected. Run this cell again and choose a file.')
else:
    filename   = list(uploaded.keys())[0]
    file_bytes = uploaded[filename]
    print(f'\n📄 File: {filename}  ({len(file_bytes)/1024:.1f} KB)')
    print()

    print('⏳ Step 1/3  Sending bill to Gemini AI for extraction...')
    try:
        bill_data = extract_bill_data(file_bytes, filename)
        print('✅ Bill data extracted.')
    except Exception as e:
        print(f'❌ Extraction failed: {e}')
        raise

    print('⏳ Step 2/3  Calculating solar recommendation...')
    solar = calculate_solar_recommendation(bill_data)
    print('✅ Solar recommendation ready.')

    print('⏳ Step 3/3  Generating Excel report...')
    try:
        excel_bytes = generate_excel(bill_data, solar)
        print('✅ Excel report generated.')
    except Exception as e:
        print(f'❌ Excel generation failed: {e}')
        raise

    print()
    print('🎉 Done! Run Step 5 below to view results and download the Excel file.')

## Step 5 — View results and download the Excel report

In [ ]:
from google.colab import files as colab_files

# Safe check — does not crash the kernel if Step 4 was not run
_ready = all(v in globals() for v in ('bill_data', 'solar', 'excel_bytes'))

if not _ready:
    print('❌ No results yet — please run Step 4 first, then re-run this cell.')
else:
    # ── Customer Info ──────────────────────────────────────────────────────
    print('=' * 60)
    print('  CUSTOMER INFORMATION')
    print('=' * 60)
    print(f"  Consumer Name      : {bill_data['consumer_name']}")
    print(f"  Consumer Number    : {bill_data['consumer_number']}")
    print(f"  Meter Number       : {bill_data.get('meter_number') or 'N/A'}")
    print(f"  Distribution Co.   : {bill_data.get('distribution_company') or 'N/A'}")
    print(f"  Billing Month      : {bill_data['billing_month']}")
    print(f"  Tariff Category    : {bill_data['tariff_category']}")

    # ── Electricity Usage ──────────────────────────────────────────────────
    print()
    print('=' * 60)
    print('  ELECTRICITY USAGE')
    print('=' * 60)
    print(f"  Units Consumed     : {bill_data['units_consumed']:,.0f} kWh")
    print(f"  Sanctioned Load    : {bill_data['sanctioned_load']} kW")
    print(f"  Total Bill Amount  : Rs. {bill_data['total_bill_amount']:,.0f}")
    print(f"  Cost per Unit      : Rs. {solar['cost_per_unit']:.2f} / kWh")
    print(f"  Avg Daily Usage    : {solar['daily_units']:.2f} kWh / day")

    # ── Solar Recommendation ───────────────────────────────────────────────
    print()
    print('=' * 60)
    print('  SOLAR SYSTEM RECOMMENDATION')
    print('=' * 60)
    print(f"  System Size        : {solar['recommended_system_size_kw']} kWp")
    print(f"  Monthly Savings    : Rs. {solar['estimated_monthly_savings']:,}")
    print(f"  Annual Savings     : Rs. {solar['estimated_annual_savings']:,}")
    print(f"  System Cost (est.) : Rs. {solar['system_cost_inr']:,}")
    print(f"  Payback Period     : {solar['payback_period_years']} years")
    print(f"  CO2 Reduction      : {solar['co2_reduction_kg_per_year']:,} kg / year")
    print(f"  25-yr Net Benefit  : Rs. {solar['net_benefit_inr']:,}")

    # ── Download Excel ─────────────────────────────────────────────────────
    print()
    print('=' * 60)
    excel_filename = f"solar_load_{bill_data['consumer_number'].replace(' ', '_')}.xlsx"
    with open(excel_filename, 'wb') as _f:
        _f.write(excel_bytes)
    print(f'📥 Downloading Excel report: {excel_filename}')
    colab_files.download(excel_filename)

---
## How it works

| Step | What happens |
|------|--------------|
| **File reading** | PDF → `pdfplumber` extracts text. Image → sent directly to Gemini Vision. |
| **AI extraction** | Gemini 1.5 Flash reads the bill and returns a structured JSON with 9 fields. |
| **Solar sizing** | `max(daily_units ÷ 4.5 sun-hrs, sanctioned_load × 0.8)` — rounded up to 0.5 kWp |
| **Savings** | Units covered by solar × cost-per-unit from the actual bill |
| **Payback** | System cost ÷ annual savings |
| **CO₂** | Annual generation × 0.82 kg/kWh (India grid emission factor, CEA 2023) |
| **Excel** | `openpyxl` creates a 4-section formatted workbook with colour-coded headers |

### AI model used
**Google Gemini 2.0 Flash** — free tier, no payment required.  
Get your key at: https://aistudio.google.com/app/apikey

---
*Energybae — Empowering People with Renewable Energy Solutions*  
*www.energybae.in | energybae.co@gmail.com | +91 9112233120*